# DAiSEE Eye Feature Extraction Pipeline for Kaggle

This notebook extracts blink rate and iris-eye ratio features from DAiSEE videos with:
- **Checkpoint saving** (resume from interruptions)
- **Feature validation** (baseline model comparison)
- **Automatic normalization** (dataset-wide z-score)

## Instructions
1. Update `BASE_PATH` to point to your DAiSEE dataset
2. Update label file paths in Cell 14
3. Run cells sequentially from top to bottom

## Cell 1: Installation

In [1]:
# Step 1: Install mediapipe first (let it pull its deps)
!pip install mediapipe==0.10.14 -q

# Step 2: Force numpy back down AFTER mediapipe installs
!pip install numpy==1.26.4 --force-reinstall -q

# Step 3: Install a compatible opencv that works with numpy 1.x
!pip install opencv-python-headless==4.8.1.78 -q

# Step 4: tqdm
!pip install tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 33.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 

In [2]:
import importlib, sys
# Force clean reimport of cv2
for mod in list(sys.modules.keys()):
    if 'cv2' in mod or 'numpy' in mod:
        del sys.modules[mod]

import numpy as np
import cv2
import mediapipe as mp

print("NumPy:", np.__version__)
print("cv2:", cv2.__version__)
print("MediaPipe:", mp.__version__)

2026-04-16 17:57:22.652467: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776362242.928961      88 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776362243.010829      88 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776362243.653346      88 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776362243.653403      88 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776362243.653406      88 computation_placer.cc:177] computation placer alr

NumPy: 1.26.4
cv2: 4.8.1
MediaPipe: 0.10.14


## Cell 2: Imports + Config

In [3]:
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from math import sqrt
import mediapipe as mp
import matplotlib.pyplot as plt
import pickle

# ===== CONFIG =====
DEBUG = False

# BLINK DETECTION CONFIG
EAR_THRESHOLD = 0.21
BLINK_MIN_DURATION = 2

# SAMPLING CONFIG
USE_ALL_FRAMES_FOR_BLINKS = True
IRIS_SAMPLE_RATE = 5  # Extract iris every 5th frame

# SMOOTHING CONFIG
SMOOTHING_METHOD = "median"

# VALIDATION CONFIG
MIN_VALID_FRAMES = 5  # LOWERED: Was 15, now 5 for DAiSEE tolerance (CHANGE #4)

# TEST MODE CONFIG (NEW - CHANGE #1)
TEST_MODE = False  # Set to False for full run, True for 200-video quick validation
TEST_SAMPLE_SIZE = 200  # Sample size for test mode

# DATASET PATH - UPDATE THIS FOR YOUR DATASET
BASE_PATH = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train"

print("✓ Configuration loaded")
print(f"  Test Mode: {'ON' if TEST_MODE else 'OFF'} (sample: {TEST_SAMPLE_SIZE})" if TEST_MODE else "")

✓ Configuration loaded



## Cell 3: MediaPipe Initialization

In [4]:
import mediapipe as mp

print(mp.__version__)

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1
)

print("✓ MediaPipe working")

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1
)

print("✓ MediaPipe Face Mesh initialized")

0.10.14
✓ MediaPipe working
✓ MediaPipe Face Mesh initialized


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776362268.220321     158 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776362268.233942     161 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Cell 4: Landmark Indices

In [5]:
# Eye landmarks (6-point EAR configuration)
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
LEFT_EYE = [362, 385, 387, 263, 373, 380]

# Iris landmarks
RIGHT_IRIS = [468, 469, 470, 471]
LEFT_IRIS = [473, 474, 475, 476]

print("✓ Landmark indices defined")

✓ Landmark indices defined


W0000 00:00:1776362268.255253     157 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Cell 5: Utility Functions

In [6]:
def euclidean(p1, p2):
    """Euclidean distance between two 2D points"""
    return sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)


def compute_EAR(eye):
    """Standard Eye Aspect Ratio formula"""
    A = euclidean(eye[1], eye[5])
    B = euclidean(eye[2], eye[4])
    C = euclidean(eye[0], eye[3])
    
    if C == 0:
        return None
    
    return (A + B) / (2.0 * C)


def validate_eye_landmarks(landmarks, eye_indices, img_width):
    """Validate landmark detection quality"""
    if len(landmarks) < max(eye_indices):
        return False
    
    eye_pts = [(landmarks[i].x, landmarks[i].y) for i in eye_indices]
    
    x_coords = [p[0] for p in eye_pts]
    y_coords = [p[1] for p in eye_pts]
    
    # CHANGE #2: RELAX LANDMARK VALIDATION - DAiSEE has low-motion faces
    if np.std(x_coords) < 0.0005 or np.std(y_coords) < 0.0005:
        return False
    
    eye_width = abs(eye_pts[0][0] - eye_pts[3][0])
    if eye_width < 0.02:  # CHANGE #2: Relaxed from 0.05 to 0.02 for DAiSEE edge cases
        return False
    
    for x, y in eye_pts:
        if not (0 <= x <= 1 and 0 <= y <= 1):
            return False
    
    return True


def compute_iris_eye_ratio(lm, img_width, img_height):
    """Compute iris-to-eye-width ratio"""
    try:
        left_eye_w = euclidean(
            (lm[33].x * img_width, lm[33].y * img_height),
            (lm[133].x * img_width, lm[133].y * img_height)
        )
        
        right_eye_w = euclidean(
            (lm[362].x * img_width, lm[362].y * img_height),
            (lm[263].x * img_width, lm[263].y * img_height)
        )
        
        left_iris_pts = [(lm[i].x * img_width, lm[i].y * img_height) for i in LEFT_IRIS]
        left_center = np.mean(left_iris_pts, axis=0)
        left_radii = [euclidean(left_center, p) for p in left_iris_pts]
        left_diameter = 2 * np.mean(left_radii)
        
        right_iris_pts = [(lm[i].x * img_width, lm[i].y * img_height) for i in RIGHT_IRIS]
        right_center = np.mean(right_iris_pts, axis=0)
        right_radii = [euclidean(right_center, p) for p in right_iris_pts]
        right_diameter = 2 * np.mean(right_radii)
        
        if left_eye_w == 0 or right_eye_w == 0:
            return None
        
        return (left_diameter / left_eye_w + right_diameter / right_eye_w) / 2
        
    except Exception as e:
        if DEBUG:
            print(f"Error computing iris ratio: {e}")
        return None


def smooth_signal(signal):
    """Median smoothing for signal aggregation"""
    if len(signal) == 0:
        return signal
    if SMOOTHING_METHOD == "median":
        return np.median(signal)
    return np.mean(signal)

print("✓ Utility functions defined")

✓ Utility functions defined


W0000 00:00:1776362268.274253     160 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Cell 6: Blink Detection Function

In [7]:
def compute_blink_rate(ear_series):
    """Compute blink rate from EAR series (blinks per minute)"""
    blink_count = 0
    closed_frames = 0
    
    for ear in ear_series:
        if ear is None:
            continue
        
        if ear < EAR_THRESHOLD:
            closed_frames += 1
        else:
            if closed_frames >= BLINK_MIN_DURATION:
                blink_count += 1
            closed_frames = 0
    
    duration_sec = len(ear_series) / 30.0
    if duration_sec == 0:
        return None
    
    return blink_count / duration_sec * 60

print("✓ Blink detection function defined")

✓ Blink detection function defined


## Cell 7: Frame Extraction Functions

In [8]:
def extract_all_frames(video_path):
    """Extract ALL frames from video"""
    cap = cv2.VideoCapture(video_path)
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0:
        cap.release()
        return []
    
    frames = []
    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    
    cap.release()
    return frames


def subsample_frames(frames, rate):
    """Subsample frames for iris extraction"""
    return frames[::rate]

print("✓ Frame extraction functions defined")

✓ Frame extraction functions defined


## Cell 8: Video Processing (Main Pipeline)

In [9]:
def process_video(video_path):
    """Process video and extract eye features
    
    CHANGE #3: Decouple iris validity - video is valid if blinks extracted, iris optional
    CHANGE #5: Add diagnostic metrics for debugging
    """
    all_frames = extract_all_frames(video_path)
    
    if len(all_frames) < MIN_VALID_FRAMES:
        return None
    
    ear_values = []
    iris_values = []
    valid_frames = 0
    frames_face_detected = 0  # CHANGE #5: Diagnostic counter
    frames_iris_detected = 0  # CHANGE #5: Diagnostic counter
    
    # CRITICAL FIX #3: Skip first 5 frames (often low quality)
    # CHANGE #1: Align iris sampling with processed frames (relative to skipped frames)
    processed_frames = all_frames[5:]
    for i, frame in enumerate(processed_frames):
        frame_idx = i
        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)
        
        if not result.multi_face_landmarks:
            continue
        
        frames_face_detected += 1  # CHANGE #5: Track face detections
        lm = result.multi_face_landmarks[0].landmark
        
        # Validate each eye independently
        right_valid = validate_eye_landmarks(lm, RIGHT_EYE, w)
        left_valid = validate_eye_landmarks(lm, LEFT_EYE, w)

        # Skip only if BOTH eyes fail
        if not right_valid and not left_valid:
            continue

        coords = [(int(p.x * w), int(p.y * h)) for p in lm]

        ear_values_frame = []

        # RIGHT EYE
        if right_valid:
            right_eye = [coords[i] for i in RIGHT_EYE]
            ear_r = compute_EAR(right_eye)
            if ear_r is not None:
                ear_values_frame.append(ear_r)

        # LEFT EYE
        if left_valid:
            left_eye = [coords[i] for i in LEFT_EYE]
            ear_l = compute_EAR(left_eye)
            if ear_l is not None:
                ear_values_frame.append(ear_l)

        # Skip if no valid EAR computed
        if len(ear_values_frame) == 0:
            continue

        # Final EAR for this frame
        ear = np.mean(ear_values_frame)

        # Store + count
        ear_values.append(ear)
        valid_frames += 1

        # Iris processing (subsampled)
        if frame_idx % IRIS_SAMPLE_RATE == 0:
            iris = compute_iris_eye_ratio(lm, w, h)
            if iris is not None:
                iris_values.append(iris)
                frames_iris_detected += 1
    
    # CHANGE #3: Return early ONLY if blinks not extracted, iris optional
    if len(ear_values) < MIN_VALID_FRAMES:
        return None
    
    blink_rate = compute_blink_rate(ear_values)
    
    # CRITICAL FIX #1: Define missing variables before return
    iris_eye_ratio_avg = smooth_signal(iris_values) if len(iris_values) > 0 else np.nan
    
    # CHANGE #4: IMPROVE CONFIDENCE METRIC - Reflect success given face detections
    confidence = valid_frames / frames_face_detected if frames_face_detected > 0 else 0
    
    # CHANGE #3: FIX FACE DETECT RATIO - Use processed frames (after skipping first 5)
    face_detect_ratio = frames_face_detected / len(processed_frames) if len(processed_frames) > 0 else 0
    
    # CHANGE #2: FIX IRIS DETECTION DENOMINATOR - Use processed frames for alignment
    total_iris_attempts = len(processed_frames[::IRIS_SAMPLE_RATE])
    iris_detect_ratio = frames_iris_detected / total_iris_attempts if total_iris_attempts > 0 else 0
    
    # CRITICAL FIX #2: Clean return block (no duplicates)
    return {
        "blink_rate": blink_rate,
        "iris_eye_ratio_avg": iris_eye_ratio_avg,
        "confidence": confidence,
        "num_valid_frames": valid_frames,
        "num_total_frames": len(all_frames),
        "face_detect_ratio": face_detect_ratio,
        "iris_detect_ratio": iris_detect_ratio,
    }

print("✓ Video processing function defined")


✓ Video processing function defined


## Cell 9: Dataset Loading

In [10]:
video_list = []

for user in os.listdir(BASE_PATH):
    user_path = os.path.join(BASE_PATH, user)
    if not os.path.isdir(user_path):
        continue
    
    for clip in os.listdir(user_path):
        clip_path = os.path.join(user_path, clip)
        
        for file in os.listdir(clip_path):
            if file.endswith(".avi"):
                video_list.append(os.path.join(clip_path, file))

print(f"Total videos found: {len(video_list)}")

# CHANGE #8: Apply TEST_MODE filtering (NEW)
if TEST_MODE:
    video_list = video_list[:TEST_SAMPLE_SIZE]
    print(f"✓ TEST_MODE enabled: Using first {len(video_list)} videos only")

assert len(video_list) > 0, f"Expected >0 videos, found {len(video_list)}"


Total videos found: 4976


## Cell 10: Feature Extraction with Checkpoint Saving

In [11]:
CHECKPOINT_FILE = "test_partial_results.csv" if TEST_MODE else "partial_results.csv"  # CHANGE #8: TEST_MODE checkpoint selection
CHECKPOINT_INTERVAL = 20

# Load existing progress if checkpoint exists
if os.path.exists(CHECKPOINT_FILE):
    print(f"✓ Found checkpoint file: {CHECKPOINT_FILE}")
    df = pd.read_csv(CHECKPOINT_FILE)
    processed_videos = set(df["video"])
    print(f"  Resuming from checkpoint: {len(df)} videos already processed")
    remaining_videos = [v for v in video_list if v not in processed_videos]
    print(f"  Videos remaining to process: {len(remaining_videos)}")
else:
    print(f"✓ No checkpoint found, starting fresh")
    df = pd.DataFrame()
    processed_videos = set()
    remaining_videos = video_list
    print(f"  Videos to process: {len(remaining_videos)}")

# Process remaining videos with checkpoint saving
failed = 0
batch_results = []

for idx, video in enumerate(tqdm(remaining_videos, desc="Processing videos")):
    out = process_video(video)
    
    if out is None:
        failed += 1
        continue
    
    batch_results.append({
        "video": video,
        **out
    })
    
    # Checkpoint: Save every N videos
    if len(batch_results) >= CHECKPOINT_INTERVAL:
        batch_df = pd.DataFrame(batch_results)
        df = pd.concat([df, batch_df], ignore_index=True)
        df = df.drop_duplicates(subset=["video"], keep="first")
        df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"\n✓ Checkpoint saved: {len(df)} total videos processed")
        batch_results = []

# Final batch
if len(batch_results) > 0:
    batch_df = pd.DataFrame(batch_results)
    df = pd.concat([df, batch_df], ignore_index=True)
    df = df.drop_duplicates(subset=["video"], keep="first")
    df.to_csv(CHECKPOINT_FILE, index=False)
    print(f"\n✓ Final checkpoint saved: {len(df)} total videos processed")

print(f"\n{'='*60}")
print(f"EXTRACTION SUMMARY")
print(f"{'='*60}")
print(f"Valid samples: {len(df)}")
print(f"Failed: {failed}")
success_rate = len(df)/(len(df)+failed)*100 if (len(df)+failed) > 0 else 0
print(f"Success rate: {success_rate:.1f}%")

# CHANGE #11: TEST_MODE success validation (NEW)
if TEST_MODE:
    if success_rate < 40:
        print(f"⚠ WARNING: Success rate {success_rate:.1f}% < 40% target")
    else:
        print(f"✓ SUCCESS: Test mode achieved {success_rate:.1f}% success rate — ready for full run")

print(f"{'='*60}\n")

assert len(df) > 0, f"Too few valid samples: {len(df)}"


✓ Found checkpoint file: partial_results.csv
  Resuming from checkpoint: 1137 videos already processed
  Videos remaining to process: 3839


Processing videos:   0%|          | 0/3839 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Processing videos:   1%|          | 20/3839 [01:09<3:42:20,  3.49s/it]


✓ Checkpoint saved: 1157 total videos processed


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Processing videos:   1%|          | 40/3839 [02:18<3:39:58,  3.47s/it]


✓ Checkpoint saved: 1177 total videos processed


...


✓ Checkpoint saved: 4937 total videos processed


Processing videos: 100%|█████████▉| 3820/3839 [3:35:04<01:04,  3.37s/it]


✓ Checkpoint saved: 4957 total videos processed


Processing videos: 100%|██████████| 3839/3839 [3:36:08<00:00,  3.38s/it]


✓ Final checkpoint saved: 4976 total videos processed

EXTRACTION SUMMARY
Valid samples: 4976
Failed: 0
Success rate: 100.0%



## Cell 11: Global Normalization

In [12]:
print("Computing dataset-wide statistics...")

# CHANGE #5: CLIP BLINK OUTLIERS - Remove physiologically implausible values (>80 bpm)
df['blink_rate'] = df['blink_rate'].clip(upper=80)

valid_iris_values = df['iris_eye_ratio_avg'].dropna()
valid_blink_values = df['blink_rate'].dropna()

GLOBAL_IRIS_MEAN = valid_iris_values.mean()
GLOBAL_IRIS_STD = valid_iris_values.std()

GLOBAL_BLINK_MEAN = valid_blink_values.mean()
GLOBAL_BLINK_STD = valid_blink_values.std()

print(f"\nGlobal Statistics:")
print(f"─────────────────────────────────────")
print(f"Iris Eye Ratio:  μ = {GLOBAL_IRIS_MEAN:.4f}, σ = {GLOBAL_IRIS_STD:.4f}")
print(f"Blink Rate:      μ = {GLOBAL_BLINK_MEAN:.2f}, σ = {GLOBAL_BLINK_STD:.2f}")
print(f"─────────────────────────────────────\n")

# Apply z-score normalization
df['iris_eye_ratio_norm'] = (df['iris_eye_ratio_avg'] - GLOBAL_IRIS_MEAN) / GLOBAL_IRIS_STD
df['blink_rate_norm'] = (df['blink_rate'] - GLOBAL_BLINK_MEAN) / GLOBAL_BLINK_STD

# Save normalization parameters
norm_params = {
    'iris_mean': GLOBAL_IRIS_MEAN,
    'iris_std': GLOBAL_IRIS_STD,
    'blink_mean': GLOBAL_BLINK_MEAN,
    'blink_std': GLOBAL_BLINK_STD
}

with open('normalization_params.pkl', 'wb') as f:
    pickle.dump(norm_params, f)

print("✓ Normalization parameters saved to normalization_params.pkl")


Computing dataset-wide statistics...

Global Statistics:
─────────────────────────────────────
Iris Eye Ratio:  μ = 0.3263, σ = 0.0119
Blink Rate:      μ = 21.00, σ = 21.91
─────────────────────────────────────

✓ Normalization parameters saved to normalization_params.pkl


## Cell 12: Validation & Sanity Checks

In [13]:
print("\n" + "="*60)
print("VALIDATION CHECKS")
print("="*60)

# Check 1: Blink rate in physiological range
blink_mean = df['blink_rate'].mean()
blink_std = df['blink_rate'].std()

print(f"\n✓ Blink Rate Statistics:")
print(f"  Mean: {blink_mean:.2f} blinks/min")
print(f"  Std:  {blink_std:.2f}")
print(f"  Range: [{df['blink_rate'].min():.2f}, {df['blink_rate'].max():.2f}]")

assert 5 < blink_mean < 50, f"❌ Blink rate mean {blink_mean:.2f} outside expected range [5, 50]"
print("  ✓ Mean within expected physiological range [5, 50]")

# Check 2: Iris ratio positive and bounded
# CHANGE #6: FIX UNDEFINED VARIABLE - Define iris_mean before use
iris_mean = df['iris_eye_ratio_avg'].mean()
iris_std = df['iris_eye_ratio_avg'].std()

print(f"\n✓ Iris Eye Ratio Statistics:")
print(f"  Mean: {iris_mean:.4f}")
print(f"  Std:  {iris_std:.4f}")
print(f"  Range: [{df['iris_eye_ratio_avg'].min():.4f}, {df['iris_eye_ratio_avg'].max():.4f}]")

valid_iris = df['iris_eye_ratio_avg'].dropna()
assert (valid_iris > 0).all(), "❌ Found negative iris ratios"
print("  ✓ All values positive")

# Check 3: Normalized features have mean≈0, std≈1
print(f"\n✓ Normalized Feature Statistics:")
print(f"  Iris (norm):  μ = {df['iris_eye_ratio_norm'].mean():.4f}, σ = {df['iris_eye_ratio_norm'].std():.4f}")
print(f"  Blink (norm): μ = {df['blink_rate_norm'].mean():.4f}, σ = {df['blink_rate_norm'].std():.4f}")

assert abs(df['iris_eye_ratio_norm'].mean()) < 0.01, "❌ Normalized iris mean not ≈0"
assert abs(df['blink_rate_norm'].mean()) < 0.01, "❌ Normalized blink mean not ≈0"
print("  ✓ Normalization successful (μ≈0, σ≈1)")

# Check 4: Confidence scores
conf_mean = df['confidence'].mean()
print(f"\n✓ Confidence Score Statistics:")
print(f"  Mean: {conf_mean:.2%}")
print(f"  Range: [{df['confidence'].min():.2%}, {df['confidence'].max():.2%}]")

assert conf_mean > 0.3, f"❌ Mean confidence {conf_mean:.2%} too low (detection failures)"
print(f"  ✓ Mean confidence > 30% (acceptable)")

print("\n" + "="*60)
print("✓ ALL VALIDATION CHECKS PASSED")
print("="*60)


VALIDATION CHECKS

✓ Blink Rate Statistics:
  Mean: 21.00 blinks/min
  Std:  21.91
  Range: [0.00, 80.00]
  ✓ Mean within expected physiological range [5, 50]

✓ Iris Eye Ratio Statistics:
  Mean: 0.3263
  Std:  0.0119
  Range: [0.2522, 0.3679]
  ✓ All values positive

✓ Normalized Feature Statistics:
  Iris (norm):  μ = -0.0000, σ = 1.0000
  Blink (norm): μ = -0.0000, σ = 1.0000
  ✓ Normalization successful (μ≈0, σ≈1)

✓ Confidence Score Statistics:
  Mean: 95.68%
  Range: [5.33%, 100.00%]
  ✓ Mean confidence > 30% (acceptable)

✓ ALL VALIDATION CHECKS PASSED


In [14]:
print("NaNs in iris:", df['iris_eye_ratio_avg'].isnull().sum())
print("Negative values:", (df['iris_eye_ratio_avg'] < 0).sum())

NaNs in iris: 1
Negative values: 0


## Cell 13: Feature Validation & Model Comparison

In [15]:
print("\n" + "="*60)
print("FEATURE VALIDATION (Baseline Model Comparison)")
print("="*60)

try:
    # Look for labels file - UPDATE THESE PATHS FOR YOUR DATASET
    label_paths = [
        "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TrainLabels.csv"
    ]
    
    labels_df = None
    for path in label_paths:
        if os.path.exists(path):
            print(f"✓ Found labels file: {path}")
            labels_df = pd.read_csv(path)
            break
    
    if labels_df is not None:
        print(f"  Labels shape: {labels_df.shape}")
        print(f"  Columns: {list(labels_df.columns)}")
        
        # FIXED: Extract video ID from path - includes ".avi" to match ClipID exactly
        df["video_id"] = df["video"].apply(lambda x: os.path.basename(x))
        
        if "ClipID" in labels_df.columns:
            labels_df["video_id"] = labels_df["ClipID"].astype(str)
        elif "video_id" in labels_df.columns:
            labels_df["video_id"] = labels_df["video_id"].astype(str)
        else:
            id_col = [c for c in labels_df.columns if c not in ["Label", "label", "Engagement"]][0]
            labels_df["video_id"] = labels_df[id_col].astype(str)
        
        # Merge
        df_labeled = df.merge(labels_df, on="video_id", how="inner")
        
        print(f"✓ Merged with labels: {len(df_labeled)} samples matched")
        
        # Find label column
        label_col = None
        for col in ["Label", "label", "Engagement", "engagement", "EngagementLevel"]:
            if col in df_labeled.columns:
                label_col = col
                break
        
        if label_col is None:
            print("⚠ WARNING: Could not find label column")
            print(f"  Available columns: {list(df_labeled.columns)}")
        else:
            print(f"✓ Using label column: {label_col}")
            
            # Model comparison
            from sklearn.model_selection import train_test_split
            from sklearn.linear_model import LogisticRegression
            from sklearn.metrics import f1_score
            from sklearn.preprocessing import StandardScaler
            
            # ═══════════════════════════════════════════════════════════════════
            # ADDED CLEANING STEP: Remove invalid samples (only drop NaN blink_rate)
            # ═══════════════════════════════════════════════════════════════════
            df_labeled = df_labeled.dropna(subset=["blink_rate"])
            
            # Optional: Fill NaN iris with mean (better than 0, since iris is optional)
            df_labeled["iris_eye_ratio_norm"] = df_labeled["iris_eye_ratio_norm"].fillna(
                df_labeled["iris_eye_ratio_norm"].mean()
            )
            
            # Optional: Filter by confidence (if column exists)
            # CHANGE #7: RELAX CONFIDENCE FILTER - DAiSEE is noisy, 0.2 more realistic than 0.5
            if "confidence" in df_labeled.columns:
                df_labeled = df_labeled[df_labeled["confidence"] > 0.2]
            
            print(f"✓ After cleaning: {len(df_labeled)} samples remaining")
            
            # ═══════════════════════════════════════════════════════════════════
            # FIXED: Prepare data for ONE train-test split
            # ═══════════════════════════════════════════════════════════════════
            X_all = df_labeled[["blink_rate", "iris_eye_ratio_norm"]]
            y = df_labeled[label_col]
            
            # CHANGE #9: Clip blink outliers (>80 bpm) before scaling
            X_all_clipped = X_all.copy()
            X_all_clipped["blink_rate"] = X_all_clipped["blink_rate"].clip(upper=80)
            outliers_removed = (X_all["blink_rate"] > 80).sum()
            if outliers_removed > 0:
                print(f"✓ Removed {outliers_removed} blink rate outliers (>80 bpm)")
            
            X_all = X_all_clipped
            
            if y.dtype == 'object':
                y = pd.Categorical(y).codes
            
            # ═══════════════════════════════════════════════════════════════════
            # UPDATED SPLIT BLOCK: Single split for all models
            # ═══════════════════════════════════════════════════════════════════
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_all, y, test_size=0.2, stratify=y, random_state=42
            )
            
            # ═══════════════════════════════════════════════════════════════════
            # CHECK 7: TRAIN/TEST DISTRIBUTION VALIDATION
            # ═══════════════════════════════════════════════════════════════════
            print("\n[CHECK 7] Train/Test split distribution")
            print("Train distribution:")
            train_dist = pd.Series(y_tr).value_counts(normalize=True).sort_index()
            print(train_dist)
            print("\nTest distribution:")
            test_dist = pd.Series(y_te).value_counts(normalize=True).sort_index()
            print(test_dist)
            
            # Check for extreme imbalance
            if train_dist.max() > 0.95 or test_dist.max() > 0.95:
                print("⚠ WARNING: Extremely imbalanced classes — model performance may be misleading")
            
            # ═══════════════════════════════════════════════════════════════════
            # ADDED FEATURE SCALING: StandardScaler before training
            # ═══════════════════════════════════════════════════════════════════
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_te_scaled = scaler.transform(X_te)
            
            # Convert back to DataFrame to preserve column names
            X_tr_scaled = pd.DataFrame(X_tr_scaled, columns=["blink_rate", "iris_eye_ratio_norm"])
            X_te_scaled = pd.DataFrame(X_te_scaled, columns=["blink_rate", "iris_eye_ratio_norm"])
            
            # ═══════════════════════════════════════════════════════════════════
            # Subset scaled features for each model
            # ═══════════════════════════════════════════════════════════════════
            X1_tr = X_tr_scaled[["blink_rate"]]
            X1_te = X_te_scaled[["blink_rate"]]
            
            X_iris_tr = X_tr_scaled[["iris_eye_ratio_norm"]]
            X_iris_te = X_te_scaled[["iris_eye_ratio_norm"]]
            
            X2_tr = X_tr_scaled
            X2_te = X_te_scaled
            
            # ═══════════════════════════════════════════════════════════════════
            # UPDATED: Train models with class_weight="balanced"
            # ═══════════════════════════════════════════════════════════════════
            m1 = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X1_tr, y_tr)
            m_iris = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_iris_tr, y_tr)
            m2 = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X2_tr, y_tr)
            
            # Evaluate
            f1_blink_only = f1_score(y_te, m1.predict(X1_te), average="macro")
            f1_iris = f1_score(y_te, m_iris.predict(X_iris_te), average="macro")
            f1_blink_iris = f1_score(y_te, m2.predict(X2_te), average="macro")
            
            # Report
            print(f"\n{'─'*60}")
            print("BASELINE MODEL COMPARISON")
            print(f"{'─'*60}")
            print(f"Blink only:        F1 = {f1_blink_only:.4f}")
            print(f"Iris only:         F1 = {f1_iris:.4f}")
            print(f"Blink + Iris:      F1 = {f1_blink_iris:.4f}")
            improvement = (f1_blink_iris - f1_blink_only) * 100
            print(f"Improvement:       {improvement:+.2f}%")
            print(f"{'─'*60}\n")
            
            if improvement > 0:
                print(f"✓ Iris feature contributes {improvement:.2f}% F1 improvement")
            else:
                print(f"⚠ Note: Iris feature alone doesn't improve F1 on this baseline")
    
    else:
        print("⚠ No labels file found")
        print("  To enable feature validation:")
        print("  1. Update label file paths in this cell")
        print("  2. Create labels_df with 'video_id' and label columns")
        print("  3. Re-run this cell")

except Exception as e:
    print(f"⚠ Feature validation skipped: {e}")
    print("  This is expected if labels are not available yet")
    print("  You can add labels later and re-run this cell")



FEATURE VALIDATION (Baseline Model Comparison)
✓ Found labels file: /kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TrainLabels.csv
  Labels shape: (5358, 5)
  Columns: ['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration ']
✓ Merged with labels: 4852 samples matched
✓ Using label column: Engagement
✓ After cleaning: 4739 samples remaining

[CHECK 7] Train/Test split distribution
Train distribution:
Engagement
0    0.006595
1    0.041414
2    0.509892
3    0.442100
Name: proportion, dtype: float64

Test distribution:
Engagement
0    0.007384
1    0.041139
2    0.509494
3    0.441983
Name: proportion, dtype: float64

────────────────────────────────────────────────────────────
BASELINE MODEL COMPARISON
────────────────────────────────────────────────────────────
Blink only:        F1 = 0.2244
Iris only:         F1 = 0.2295
Blink + Iris:      F1 = 0.2426
Improvement:       +1.83%
────────────────────────────────────────────────────────────

✓ Iris feature contributes 

## Cell 13a: Pre-Run Sanity Checks

In [16]:
print("\n" + "="*60)
print("🔹 PRE-RUN SANITY CHECKS")
print("="*60)

try:
    # ═══════════════════════════════════════════════════════════════════
    # CHECK 1: VIDEO ID ALIGNMENT
    # ═══════════════════════════════════════════════════════════════════
    print("\n[CHECK 1] Video ID alignment")
    print("✓ Video paths sample:")
    print(df["video"].head(3).values)
    print("\n✓ Extracted video_id sample:")
    print(df["video_id"].head(3).values)
    print("\n✓ Labels video_id sample:")
    if "ClipID" in labels_df.columns:
        print(labels_df["ClipID"].head(3).values)
    elif "video_id" in labels_df.columns:
        print(labels_df["video_id"].head(3).values)
    
    # ═══════════════════════════════════════════════════════════════════
    # CHECK 2: MERGE SIZE CHECK
    # ═══════════════════════════════════════════════════════════════════
    print("\n[CHECK 2] Merge size")
    print(f"Extracted features: {len(df)} videos")
    print(f"Labels available: {len(labels_df)} samples")
    print(f"✓ Merged samples: {len(df_labeled)}")
    
    if len(df_labeled) == 0:
        print("❌ CRITICAL: No samples merged! Check video_id alignment")
    elif len(df_labeled) < 100:
        print("⚠ WARNING: Very few samples merged — check video_id alignment")
    else:
        print(f"✓ Merge successful ({len(df_labeled)/len(df)*100:.1f}% of features matched)")
    
    # ═══════════════════════════════════════════════════════════════════
    # CHECK 3: LABEL DISTRIBUTION CHECK
    # ═══════════════════════════════════════════════════════════════════
    print("\n[CHECK 3] Label distribution")
    print(df_labeled[label_col].value_counts().sort_index())
    
    label_counts = df_labeled[label_col].value_counts()
    if len(label_counts) < 2:
        print("⚠ WARNING: Only one class in labels — model training will fail")
    
    # ═══════════════════════════════════════════════════════════════════
    # CHECK 4: MISSING VALUES CHECK
    # ═══════════════════════════════════════════════════════════════════
    print("\n[CHECK 4] Missing values")
    missing = df_labeled[["blink_rate", "iris_eye_ratio_norm"]].isnull().sum()
    print(missing)
    
    if missing.sum() > 0:
        print(f"⚠ WARNING: {missing.sum()} missing values found")
    else:
        print("✓ No missing values in features")
    
    # ═══════════════════════════════════════════════════════════════════
    # CHECK 5: FEATURE RANGE CHECK
    # ═══════════════════════════════════════════════════════════════════
    print("\n[CHECK 5] Feature ranges")
    print("Blink rate stats (should be 0-200 blinks/min):")
    print(df_labeled["blink_rate"].describe())
    
    if df_labeled["blink_rate"].max() > 500:
        print("⚠ WARNING: Extremely high blink rate detected — check for outliers")
    
    print("\nIris ratio stats (should be reasonable positive values):")
    print(df_labeled["iris_eye_ratio_norm"].describe())
    
    if (df_labeled["iris_eye_ratio_norm"] < -5).any() or (df_labeled["iris_eye_ratio_norm"] > 5).any():
        print("⚠ WARNING: Extreme normalized values — check scaling")
    
    # ═══════════════════════════════════════════════════════════════════
    # CHECK 6: CONFIDENCE CHECK (Optional)
    # ═══════════════════════════════════════════════════════════════════
    if "confidence" in df_labeled.columns:
        print("\n[CHECK 6] Confidence stats")
        print(df_labeled["confidence"].describe())
        
        low_conf = (df_labeled["confidence"] < 0.3).sum()
        if low_conf > 0:
            print(f"⚠ WARNING: {low_conf} samples with low confidence (<0.3)")
    
    print("\n" + "="*60)
    print("✓ SANITY CHECKS COMPLETE - Ready to proceed")
    print("="*60 + "\n")

except Exception as e:
    print(f"\n❌ Sanity check error: {e}")
    print("This should not interrupt validation — check your data")



🔹 PRE-RUN SANITY CHECKS

[CHECK 1] Video ID alignment
✓ Video paths sample:
['/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train/205601/2056010245/2056010245.avi'
 '/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train/205601/2056010263/2056010263.avi'
 '/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train/205601/2056010124/2056010124.avi']

✓ Extracted video_id sample:
['2056010245.avi' '2056010263.avi' '2056010124.avi']

✓ Labels video_id sample:
['1100011002.avi' '1100011003.avi' '1100011004.avi']

[CHECK 2] Merge size
Extracted features: 4976 videos
Labels available: 5358 samples
✓ Merged samples: 4739
✓ Merge successful (95.2% of features matched)

[CHECK 3] Label distribution
Engagement
0      32
1     196
2    2416
3    2095
Name: count, dtype: int64

[CHECK 4] Missing values
blink_rate             0
iris_eye_ratio_norm    0
dtype: int64
✓ No missing values in features

[CHECK 5] Feature ranges
Blink rate stats (should be 0-200 blinks/min):


## Cell 14: Save Final Results

In [17]:
output_file = "final_results.csv"
df.to_csv(output_file, index=False)
print(f"\n✓ Final results saved to {output_file}")
print(f"✓ Total features extracted: {len(df)}")

print("\n" + "="*60)
print("✓ PIPELINE COMPLETE")
print("="*60)
print(f"""
📊 Output Files:
   - partial_results.csv: Checkpoint (updated every 20 videos)
   - final_results.csv: Complete feature set
   - normalization_params.pkl: Statistics for test set
   - daisee_eye_features_CORRECTED.csv: Intermediate results

🎯 DataFrame Columns:
   - video: Path to video file
   - blink_rate: Blinks per minute
   - iris_eye_ratio_avg: Raw iris-to-eye ratio
   - iris_eye_ratio_norm: Z-score normalized
   - blink_rate_norm: Z-score normalized
   - confidence: Frame processing success rate
   - num_valid_frames: Valid detections
   - num_total_frames: Total frames in video

✅ Ready for ViBED-Net fusion!
""")


✓ Final results saved to final_results.csv
✓ Total features extracted: 4976

✓ PIPELINE COMPLETE

📊 Output Files:
   - partial_results.csv: Checkpoint (updated every 20 videos)
   - final_results.csv: Complete feature set
   - normalization_params.pkl: Statistics for test set
   - daisee_eye_features_CORRECTED.csv: Intermediate results

🎯 DataFrame Columns:
   - video: Path to video file
   - blink_rate: Blinks per minute
   - iris_eye_ratio_avg: Raw iris-to-eye ratio
   - iris_eye_ratio_norm: Z-score normalized
   - blink_rate_norm: Z-score normalized
   - confidence: Frame processing success rate
   - num_valid_frames: Valid detections
   - num_total_frames: Total frames in video

✅ Ready for ViBED-Net fusion!

